# Crivo — treinar o pontuador de frases (modelo de linguagem maior) em GPU

O Crivo monta várias formas de dizer a mesma resposta, um verificador descarta as que inventam ou perdem
algum fato, e um **modelo de linguagem** escolhe a forma mais natural. O modelo atual é pequeno
(2,6 milhões de parâmetros, treinado em CPU) e acerta só 76% (dev) e 68% (retido) na bateria de fluência.

Este notebook treina, **do zero**, um modelo maior (≈ 8,7 milhões de parâmetros, 6 camadas) com os mesmos dados
públicos do projeto: Wikipédia em português (CC BY-SA), diálogos humanos do OpenAssistant 2 (Apache-2.0) e
diálogos públicos do Tucano-SFT. Nenhum peso de modelo externo é usado.

**Tempo:** cerca de 1h30 numa GPU T4 do Colab (gratuita).
Ative a GPU em **Ambiente de execução → Alterar o tipo de ambiente de execução → T4 GPU** e rode as células em ordem.

No fim, o notebook compara o modelo novo com o atual na bateria de fluência e gera `pontuador_pt.zip` para você me enviar.
O modelo novo só entra no Crivo se acertar mais que o atual — eu confiro também no conjunto retido.

In [ ]:
# 1. Conferir a GPU
!nvidia-smi -L || echo "Sem GPU: ative em Ambiente de execução → Alterar o tipo de ambiente de execução"
import torch
print("torch", torch.__version__, "| GPU disponível:", torch.cuda.is_available())

In [ ]:
# 2. Baixar o código do Crivo
BRANCH = "main"  #@param {type:"string"}
!rm -rf CRIVO && git clone --depth 1 -b "$BRANCH" https://github.com/HROSONE/CRIVO.git
%cd CRIVO
!git log --oneline -1
!pip -q install "tokenizers>=0.20,<1" "pyarrow>=15,<26"

In [ ]:
# 3. Baixar os dados públicos nas revisões fixadas (o script confere o SHA-256 de cada arquivo)
!python scripts/baixar_fontes_linguagem.py --saida trabalho/fontes --dialogos-publicos

In [ ]:
# 4. Preparar o corpus (documentos e diálogos separados em treino, validação e teste)
DOCUMENTOS = 60000  #@param {type:"integer"}
!python scripts/preparar_linguagem_profunda.py --wikipedia trabalho/fontes/wikipedia.pt.parquet \
    --origem-wikipedia dados/origem_wikipedia_20261001.json --oasst2 trabalho/fontes/oasst2.pt.jsonl \
    --dialogos-publicos trabalho/fontes/tucano-sft.parquet --limite-dialogos-publicos 80000 \
    --saida trabalho/corpus --documentos $DOCUMENTOS --vocabulario 4096 --contexto 256

In [ ]:
# 5. Aprender português do zero (pré-treino)
PASSOS_LINGUAGEM = 30000  #@param {type:"integer"}
DIMENSAO = 320  #@param {type:"integer"}
CAMADAS = 6  #@param {type:"integer"}
CABECAS = 8  #@param {type:"integer"}
!python -u scripts/treinar_linguagem_profunda.py --corpus trabalho/corpus --saida trabalho/pretreino \
    --fase linguagem --passos $PASSOS_LINGUAGEM --lote 32 --dimensao $DIMENSAO --camadas $CAMADAS \
    --cabecas $CABECAS --avaliar-a-cada 2000 --salvar-a-cada 1000 --dispositivo cuda

In [ ]:
# 6. Aprender respostas de conversa (diálogos com papéis)
PASSOS_DIALOGO = 6000  #@param {type:"integer"}
!python -u scripts/treinar_linguagem_profunda.py --corpus trabalho/corpus --saida trabalho/dialogo \
    --fase dialogo --inicial trabalho/pretreino --passos $PASSOS_DIALOGO --lote 32 --lr 0.0003 \
    --dimensao $DIMENSAO --camadas $CAMADAS --cabecas $CABECAS --avaliar-a-cada 1000 --salvar-a-cada 500 \
    --dispositivo cuda

In [ ]:
# 7. Exportar para NumPy e comparar com o pontuador atual na bateria de fluência (dev)
!python scripts/exportar_pontuador.py --origem trabalho/dialogo --destino artefatos/pontuador_pt
!python scripts/avaliar_fluencia.py dev --pasta artefatos/linguagem_profunda
!python scripts/avaliar_fluencia.py dev --pasta artefatos/pontuador_pt

In [ ]:
# 8. Baixar o modelo novo (pesos em NumPy, tokenizador e relatório do treino)
!cp trabalho/dialogo/pesos.pt artefatos/pontuador_pt/pesos.pt
!cd artefatos && zip -r ../pontuador_pt.zip pontuador_pt && cd .. && ls -la pontuador_pt.zip
from google.colab import files
files.download("pontuador_pt.zip")

### Depois do treino
Me envie o `pontuador_pt.zip`. Eu confiro a fluência também no conjunto retido, comparo a paridade
NumPy × PyTorch, rodo todas as baterias e a suíte, e só então troco o pontuador.

Se a sessão do Colab cair no meio do treino, rode de novo a célula do passo 5 ou 6 acrescentando `--retomar`
ao comando: o treino continua do último checkpoint salvo.